# Coupon Experiment Decision Framework — Walkthrough

This notebook walks through the framework stage by stage using simulated data.

Note: the dataset contains hidden latent fields for later validation. Stages 1-4 only use observable fields; latent truth and long-term outcomes are revealed in Stage 5.

> **Data note:** The simulated dataset includes latent fields for validation
> purposes. The walkthrough treats those fields as unavailable until the
> calibration stage. Stages 1 to 4 use only observable fields.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/simulated_coupon_experiment.csv")

# Stages 1-4 only use observable fields. The CSV also contains latent ground
# truth and long-term outcome columns, which are intentionally not loaded or
# shown until Stage 5.
observable_cols = [
    "user_id",
    "group",
    "coupon_received",
    "signup_day",
    "exposure_day",
    "made_first_purchase_30d",
    "days_to_first_purchase",
    "repeat_visits_14d",
    "browse_sessions_14d",
    "product_views_14d",
    "cart_adds_14d",
    "discount_page_views_14d",
    "coupon_used",
]

df[observable_cols].head()

## Stage 1: Measurement Design

Stage 1 is about designing the experiment so it can answer a stronger question than "did the 30-day first-purchase rate go up?" — it should also capture what *kind* of value any lift represents. That means committing up front to the primary metric, guardrail metrics, holdout logic, observation windows, and the post-period tracking needed to later assess incremental demand, customer quality, and borrowed-demand risk.

### Trap 1: The Headline Lift

In [ ]:
rates = (
    df.groupby("group")["made_first_purchase_30d"]
    .mean()
    .rename("first_purchase_rate_30d")
)

control_rate = rates.loc["control"]
treatment_rate = rates.loc["treatment"]
lift_pp = (treatment_rate - control_rate) * 100

print(f"Control   30d first-purchase rate: {control_rate:.2%}")
print(f"Treatment 30d first-purchase rate: {treatment_rate:.2%}")
print(f"Headline lift:                     {lift_pp:+.2f}pp")

rates.to_frame()

The coupon appears to improve the 30-day first-purchase rate. But this only answers whether the rate went up. It does not tell us what kind of value that lift represents.

### Trap 2: Same Lift, Different Reality

| Scenario | Headline lift | Dominant hidden reality | Decision implication |
|----------|---------------|-------------------------|----------------------|
| A | +6pp | More persuadable high-value users | Consider scale |
| B | +6pp | More discount-driven and pulled-forward users | Adjust or stop |

Two experiments can show the same headline lift while representing very different underlying realities. The same +6pp could justify scaling in one case and stopping in another. This is the gap the rest of the framework aims to close. We will return to validate this with long-term outcomes in Stage 5.

## Stage 2: Early Signal Layer

While long-term outcomes are still immature, we turn raw early behaviour into
a small set of transparent, rule-based **derived signals**. Each signal is a
simple combination of *observable* fields only, expressed on a common 0–1
percentile scale so that fields with different units stay comparable:

- **engagement_intensity** — breadth of early activity (repeat visits, browse
  sessions, product views).
- **purchase_intent** — depth of intent (cart adds), kept separate so product
  views are not weighted twice.
- **discount_dependency** — reliance on the discount itself (discount-page
  views combined with whether a coupon was redeemed).
- **early_timing** — how early the first purchase landed (purchasers only; NaN
  otherwise). On its own this is *not* a borrowed-demand signal.

In [ ]:
import sys

sys.path.insert(0, "../src")
from signals import add_derived_signals

# Stage 2 reads observable fields only.
signals_df = add_derived_signals(df[observable_cols])

derived_cols = [
    "engagement_intensity",
    "purchase_intent",
    "discount_dependency",
    "early_timing",
]

# Show the observable inputs alongside the derived signals only - latent and
# long-term columns are never loaded here.
signals_df[["user_id", "group", "made_first_purchase_30d"] + derived_cols].head()

Each derived signal is a percentile within the cohort, so it reads as *where a
user sits relative to everyone else* rather than a raw count. High
`engagement_intensity` and `purchase_intent` point toward genuine interest;
high `discount_dependency` points toward discount-led behaviour; `early_timing`
simply flags an early purchase, which only becomes meaningful **in combination**
with the other signals in Stage 3.

## Stage 3: Early Read Model

Stage 3 turns the early signals into an initial read on customer quality and
borrowed-demand risk. This component is **intentionally modular**: the v1 below
is a transparent, rule-based read, and it could later be swapped for a
statistical or ML model without changing the rest of the framework, as long as
it emits the same read columns.

It follows a "score, then band" shape. Continuous scores are computed
internally as an intermediate step, then exposed as a coarse **high / medium /
low** read using within-cohort tertiles — no hand-picked thresholds, and no
false precision.

- **quality_score** — high engagement + high intent + low discount dependency.
- **borrowed_risk_score** — early purchase **and** weak engagement **and** high
  discount dependency, combined as a product so *all three* must be present.
  Buying early on its own does not raise it. Purchasers only.
- **discount_dependency_score** — the `discount_dependency` signal directly.

In [ ]:
from early_read import add_early_read

read_df = add_early_read(signals_df)

score_cols = ["quality_score", "borrowed_risk_score", "discount_dependency_score"]
read_cols = ["quality_read", "borrowed_risk_read", "discount_dependency_read"]

# Internal scores are intermediate; the high/medium/low reads are what we act on.
read_df[["user_id", "group"] + score_cols + read_cols].head()

### Cohort-level Read

The read is summarised at the **cohort level**, not per user. We compare the
*mix* of high / medium / low reads among **purchasers** in each arm. This keeps
the output a decision-useful read on the cohort rather than an individual-level
label or prediction.

In [ ]:
purchasers = read_df[read_df["made_first_purchase_30d"] == 1]
levels = ["high", "medium", "low"]

parts = []
for col in read_cols:
    mix = (
        purchasers.groupby("group")[col]
        .value_counts(normalize=True)
        .unstack("group")
        .reindex(index=levels)
    )
    mix.index = pd.MultiIndex.from_product([[col], mix.index], names=["read", "level"])
    parts.append(mix)

# Share (%) of each arm's purchasers falling in each read band, plus the gap.
cohort_summary = pd.concat(parts) * 100
cohort_summary["delta_pp"] = cohort_summary["treatment"] - cohort_summary["control"]
cohort_summary.round(1)

**Reading the cohort.** Compared with control purchasers, treatment purchasers
show a much higher share of **high** discount-dependency reads (about 73% vs
21%) and **high** borrowed-risk reads (about 51% vs 11%), alongside a much
lower share of **high**-quality reads (about 36% vs 70%). Taken together, that
is a decision-useful read that a meaningful part of the headline lift may be
carried by discount-led, pulled-forward purchases rather than durable
incremental demand.

This stays at the cohort level on purpose. It is a read on the *mix* of the
treatment purchaser cohort — not a label on any individual user, and not a
prediction of any one user's long-term value. It points to where the risk
likely sits. Stage 4 turns this into a scale / adjust / stop recommendation,
and Stage 5 checks the read against mature long-term outcomes.

## Stage 4: Decision Tree

Stage 4 upgrades the earlier "decision mapping" idea into an explicit **decision
tree**: the experiment passes through ordered **gates**, and each gate either
stops the flow or hands control to the next. This makes the reasoning auditable —
you can see exactly where a decision was made and why.

- **Gate 0 — Experiment Readiness and Data Quality Check.** Computable checks on
  sample ratio, duplicate users, missingness, coupon logic, timing validity, and
  metric availability. Any failure stops the flow: *do not decide on bad data.*
- **Gate 1 — Primary Metric Check.** Did the 30-day first-purchase rate move? This
  only asks whether the short-term target moved, not whether the lift is durable
  or valuable.
- **Gate 2 — Guardrail and Dependency Check.** Discount dependency as an *early
  risk proxy* — not a final cost metric (true cost is settled in Stage 5).
- **Gate 3 — Early Quality Read.** The treatment purchaser quality *mix*.
- **Gate 4 — Borrowed-Demand Risk Read.** The treatment purchaser borrowed-risk
  *mix* (a combined signal, not "bought early" alone).
- **Gate 5 — Decision Output.** Combine the reads into: Scale · Scale selectively
  / Adjust · Adjust · Continue measurement · Stop.

Stage 4 still uses only observable fields and the Stage 3 reads — no latent or
long-term outcome fields.

In [ ]:
from decision_tree import run_decision_tree, format_report

# Stage 4 runs on the Stage 3 read dataframe: observable fields + early reads only.
result = run_decision_tree(read_df)
print(format_report(result))

**Reading this case.** Gate 0 passes (data quality acceptable). Gate 1: the
primary lift exists (**+5.84pp**). Gate 2: discount dependency is **high** (about
73% of treatment purchasers read high dependency vs 21% of control). Gate 3: the
treatment purchaser quality mix is **weaker** (about 36% high-quality vs 70% in
control). Gate 4: the borrowed-risk read is **higher** (about 51% vs 11%).

**Decision: Adjust, not broad scale.** Do not scale broadly yet. Adjust targeting
or incentive design, keep a holdout, and continue tracking long-term value.

The lift is real, so this is not a *Stop*. But the early reads say a meaningful
share of the lift is carried by discount-led, pulled-forward purchases rather than
durable incremental demand — an unfavorable quality/risk mix — so it is not a
broad *Scale* either. The defensible move is to refine the offer and targeting,
hold out a control, and let long-term value mature before any broad rollout.

## Framework Flow

The structure stays fixed across experiments; the method inside each stage can
improve over time. Stage 4 is where the flow branches into a decision, and
Stage 5 feeds what it learns back into the next cycle.

```
Stage 1  Measurement Design
   |  primary metric, guardrails, holdout, observation windows, post-period tracking
   v
Stage 2  Early Signal Layer
   |  observable behaviour -> transparent derived signals (0-1 percentile scale)
   v
Stage 3  Early Read Model
   |  signals -> high / medium / low reads (quality, borrowed risk, discount dependency)
   v
Stage 4  Decision Tree
   |  Gate 0  data quality ----fail----> do not decide; fix instrumentation
   |  Gate 1  primary lift ----none----> Stop / redesign offer
   |  Gate 2  discount dependency (early risk proxy)
   |  Gate 3  quality mix
   |  Gate 4  borrowed-risk mix
   |  Gate 5  combine --> Scale | Scale selectively / Adjust | Adjust | Continue | Stop
   v
Stage 5  Backtest, Calibration & Monitoring
   |  5A backtest reads vs mature long-term outcomes
   |  5B calibrate the next cycle (update signals / gates / targeting)
   |  5C monitor after rollout (future)
   v
Next experiment  (improved signals, gates, and targeting)
```

## Stage 5: Backtest, Calibration, and Monitoring Loop

Stage 5 closes the loop once long-term outcomes mature. It has three parts:

- **5A — Backtest.** Compare the Stage 3 early reads against mature long-term
  outcomes: did the cohorts we read early as high quality / high risk / high
  dependency actually behave that way?
- **5B — Calibration.** Turn the backtest findings into concrete updates for the
  next experiment cycle.
- **5C — Monitoring.** Track the chosen metrics continuously after a change ships.

This commit covers **5A and 5B**. Monitoring (5C) comes in a later commit.

### Stage 5A: Backtest — Do Early Reads Align with Later Outcomes?

This is the **first** point in the walkthrough where we open the long-term
outcome fields (`repeat_purchase_180d`, `orders_180d`, `net_revenue_180d`,
`realized_ltv_180d`, …). Stages 1–4 never loaded them.

The backtest groups purchasers by their early read **band** and looks at the
average long-term outcome in each band. The question is whether the early reads
**align** with what actually happened — a directional check on the read
*instrument*, not a claim that the reads *predict* any individual user's outcome.

For the borrowed-risk table we first derive **post-period orders**:

```
post_period_orders_31_180d = orders_180d − made_first_purchase_30d
```

`orders_180d` includes the first purchase, so we subtract it to isolate demand
that landed *after* the 30-day first-purchase window — the part borrowed demand
is really about.

*Cohort: all purchasers, both arms pooled.*

In [ ]:
from backtest import run_backtest, LONG_TERM_FIELDS

# Stage 5 opens the long-term fields and joins them onto the Stage 3 reads.
# (run_backtest derives post_period_orders_31_180d internally.)
backtest_df = read_df.merge(df[["user_id", *LONG_TERM_FIELDS]], on="user_id")
tables = run_backtest(backtest_df)

print("Table 1 — Quality read vs long-term value")
display(tables["quality_vs_value"])
print("Table 2 — Borrowed-risk read vs long-term outcome")
display(tables["borrowed_risk_vs_outcome"])
print("Table 3 — Discount-dependency read vs long-term value")
display(tables["discount_dependency_vs_value"])

**Reading the three tables (all directional, not predictive).**

- **Table 1 — quality read vs value.** The high-quality-read band shows the
  highest repeat rate, order count, and realized LTV, stepping down through medium
  to low. The early quality read *aligns* with later value.
- **Table 2 — borrowed-risk read vs outcome.** The high-risk band has the
  **weakest** post-period (day 31–180) orders and the lowest net revenue / LTV,
  while the low-risk band has the strongest. The borrowed-risk read *aligns* with
  weaker post-period demand.
- **Table 3 — discount-dependency read vs value.** The high-dependency band shows
  the **lowest** repeat rate and net revenue / LTV. High discount dependency
  *aligns* with weaker long-term value.

Together these *validate the direction* of the early reads on this dataset: the
cohorts flagged early went on to behave as flagged. This is consistency with
later outcomes, not proof of a predictive model — but it is enough to justify
keeping these reads, and (per Stage 5B) strengthening the discount-dependency
guardrail, in the next cycle.

### Stage 5B: Calibration — Updating the Next Cycle

Calibration turns what the backtest revealed into concrete changes for the next
experiment. This is the loop closing: each finding maps to an update and the
stage it affects. This is a *conceptual* update table — no automatic weight
tuning or threshold fitting.

| Backtest finding | Next-cycle update | Stage affected |
|---|---|---|
| High discount dependency aligns with weaker LTV | Promote discount dependency to a stronger guardrail | Stage 1 / 4 |
| Early timing alone is insufficient | Keep timing only as a combined risk signal | Stage 3 |
| Engagement offsets timing risk | Preserve engagement as a risk reducer | Stage 3 |
| Treatment lift comes with weaker quality mix | Adjust targeting before broad scale | Stage 4 |
| Observable fields lack post-purchase engagement | Add post-purchase engagement tracking | Stage 1 / 2 |
| Long-term outcomes mature slowly | Keep holdout and define follow-up windows | Stage 1 / 5 |

Over repeated cycles this turns the framework into a learning system: signals
that aligned with durable value are reinforced, signals that did not are demoted
or replaced, and the decision tree's gates get sharper.

In [ ]:
from backtest import next_cycle_updates

# The same calibration table as a data structure (exercised here; rendered above).
next_cycle_updates()